# 03c. Limpieza y diagnóstico de `artists.csv` para Hidden Gems v3

Este notebook prepara la tabla de artistas para enriquecer la definición de canciones tipo *hidden gem*.

En la versión anterior, `hidden_gems_v2_modern`, se detectó que algunas canciones con baja popularidad a nivel track pertenecían a artistas extremadamente famosos, como J Balvin, Maroon 5, Lady Gaga, Justin Bieber o Daddy Yankee. Esto sugiere que la variable `popularity` de tracks no necesariamente representa la fama global del artista ni de la canción en sentido general, sino la popularidad de una fila, versión o identificador específico dentro del dataset.

Por ello, se incorpora información de `artists.csv`, principalmente:

- popularidad del artista;
- número de seguidores del artista;
- géneros asociados;
- número de géneros.

El objetivo es construir una tabla limpia y defendible para realizar joins con tracks y evitar que canciones de artistas masivos sean clasificadas como *hidden gems* solo porque tienen baja popularidad a nivel track.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType, StringType

In [2]:
spark = (
    SparkSession.builder
    .appName("03c_artists_cleaning_for_hidden_gems")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 02:45:46 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/21 02:45:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 02:45:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
RAW_DIR = "../raw"
PROCESSED_DIR = "../processed"

ARTISTS_RAW_PATH = f"{PROCESSED_DIR}/artists_raw.parquet"

ARTISTS_CLEAN_PATH = f"{PROCESSED_DIR}/artists_clean.parquet"
ARTISTS_REF_PATH = f"{PROCESSED_DIR}/artists_ref.parquet"

ARTISTS_DIAG_DIR = f"{PROCESSED_DIR}/diagnostics/artists"

## 1. Carga de datos

Se carga la versión Parquet de `artists.csv`. El uso de Parquet permite trabajar de forma más eficiente que con CSV, ya que conserva tipos de datos, permite compresión y mejora el desempeño de lectura en Spark.

A partir de este punto, el procesamiento principal se realiza con PySpark, respetando la restricción tecnológica del proyecto de Datos Masivos.

In [4]:
artists_raw = spark.read.parquet(ARTISTS_RAW_PATH).cache()

In [5]:
artists_raw.printSchema()

root
 |-- id: string (nullable = true)
 |-- followers: double (nullable = true)
 |-- genres: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)



In [6]:
artists_raw.show(10, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+----------------------+---------+------+----------------------------------+----------+
|id                    |followers|genres|name                              |popularity|
+----------------------+---------+------+----------------------------------+----------+
|31Sb76IFcQBvnSTP0BrYOo|4.0      |[]    |Bobby Diamond                     |0         |
|30K9bUO2UVuecuttUdnVQp|3.0      |[]    |The Quickest Way Out              |0         |
|6eMnOoMBmtMesYaS8T2R5G|75.0     |[]    |Wigan's Ovation                   |10        |
|0Y3V9aEmAL78ePgFsS5FnD|7.0      |[]    |Dianne Steinberg                  |0         |
|2B0gsVlWByBbij8h6y2TFo|33.0     |[]    |The Patterson Twins               |0         |
|404qQMseMZFa7yxJFHAqeF|5.0      |[]    |Tommy & Eddie                     |0         |
|4j2stfqVSryQAeEuhUyZVa|8.0      |[]    |Luv Co                            |0         |
|1gOvuBeTA5WDWtzoDSLAhB|108.0    |[]    |Lee Mcdonald aka: Cleveland Parker|8         |
|76nrx0hcuOEP3R9ZgUeJbt|0.0     

In [7]:
total_artists_raw = artists_raw.count()
print(f"Total de filas en artists_raw: {total_artists_raw:,}")

[Stage 2:====================================>                      (5 + 3) / 8]

Total de filas en artists_raw: 1,162,095


In [8]:
from pyspark.sql import functions as F

In [9]:
artists_typed = (
    artists_raw
    .select(
        F.trim(F.col("id").cast("string")).alias("id"),
        F.col("followers").cast("double").alias("followers"),
        F.col("genres").cast("string").alias("genres_raw"),
        F.trim(F.col("name").cast("string")).alias("name"),
        F.col("popularity").cast("int").alias("popularity")
    )
    .cache()
)

In [10]:
artists_typed.printSchema()

root
 |-- id: string (nullable = true)
 |-- followers: double (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)



In [11]:
artists_typed.show(10, truncate=False)

+----------------------+---------+----------+----------------------------------+----------+
|id                    |followers|genres_raw|name                              |popularity|
+----------------------+---------+----------+----------------------------------+----------+
|31Sb76IFcQBvnSTP0BrYOo|4.0      |[]        |Bobby Diamond                     |0         |
|30K9bUO2UVuecuttUdnVQp|3.0      |[]        |The Quickest Way Out              |0         |
|6eMnOoMBmtMesYaS8T2R5G|75.0     |[]        |Wigan's Ovation                   |10        |
|0Y3V9aEmAL78ePgFsS5FnD|7.0      |[]        |Dianne Steinberg                  |0         |
|2B0gsVlWByBbij8h6y2TFo|33.0     |[]        |The Patterson Twins               |0         |
|404qQMseMZFa7yxJFHAqeF|5.0      |[]        |Tommy & Eddie                     |0         |
|4j2stfqVSryQAeEuhUyZVa|8.0      |[]        |Luv Co                            |0         |
|1gOvuBeTA5WDWtzoDSLAhB|108.0    |[]        |Lee Mcdonald aka: Cleveland Parker|

In [12]:
total_artists_typed = artists_typed.count()

print(f"Total de filas en artists_raw:   {total_artists_raw:,}")
print(f"Total de filas en artists_typed: {total_artists_typed:,}")

[Stage 7:==============>                                            (2 + 6) / 8]

Total de filas en artists_raw:   1,162,095
Total de filas en artists_typed: 1,162,095


## IDs únicos y duplicados

In [13]:
total_rows = artists_typed.count()

unique_ids = (
    artists_typed
    .filter(F.col("id").isNotNull() & (F.col("id") != ""))
    .select("id")
    .distinct()
    .count()
)

print(f"Total de filas: {total_rows:,}")
print(f"IDs únicos no nulos: {unique_ids:,}")
print(f"Filas extra respecto a IDs únicos: {total_rows - unique_ids:,}")

[Stage 14:=====================>                                    (3 + 5) / 8]

Total de filas: 1,162,095
IDs únicos no nulos: 1,162,095
Filas extra respecto a IDs únicos: 0


## Diagnóstico de identificadores

La columna `id` funciona como identificador único del artista.  
El diagnóstico muestra que el número total de filas coincide con el número de IDs únicos no nulos.

Por lo tanto, no se detectan IDs faltantes ni duplicados. Esto es importante porque `id` será la llave para unir la tabla de artistas con la tabla de canciones. Al no existir duplicados, el join no debería multiplicar artificialmente las filas de tracks.

In [14]:
#nulos por columna:

In [15]:
nulls_summary = artists_typed.select(
    F.sum((F.col("id").isNull() | (F.col("id") == "")).cast("int")).alias("id_null_or_empty"),
    F.sum(F.col("followers").isNull().cast("int")).alias("followers_null"),
    F.sum(F.col("genres_raw").isNull().cast("int")).alias("genres_null"),
    F.sum((F.col("name").isNull() | (F.col("name") == "")).cast("int")).alias("name_null_or_empty"),
    F.sum(F.col("popularity").isNull().cast("int")).alias("popularity_null")
)

nulls_summary.show(truncate=False)

+----------------+--------------+-----------+------------------+---------------+
|id_null_or_empty|followers_null|genres_null|name_null_or_empty|popularity_null|
+----------------+--------------+-----------+------------------+---------------+
|0               |11            |0          |0                 |0              |
+----------------+--------------+-----------+------------------+---------------+



### Diagnóstico de nulos

El diagnóstico de valores faltantes muestra que no existen IDs nulos, nombres vacíos, géneros nulos ni popularidad nula. Esto indica que la tabla de artistas es bastante completa para los campos principales.

Solo se detectan 11 registros con `followers` nulo. Dado que representan una proporción mínima del total de artistas, no se eliminan en esta etapa. La limpieza se mantiene conservadora para evitar pérdida innecesaria de información.

Sin embargo, estos casos se deben considerar al construir `hidden_gems_v3`, ya que el filtro por seguidores requiere comparar contra un umbral numérico.

## Rangos de followers y popularidad

Se revisan las dos variables numéricas centrales de `artists.csv`:

- `followers`: número de seguidores del artista.
- `popularity`: popularidad del artista en escala de 0 a 100.

Este diagnóstico permite detectar valores fuera de rango y entender la distribución real de los artistas. En particular, `followers` suele tener una distribución muy asimétrica: la mayoría de artistas tiene pocos seguidores y unos pocos concentran millones.

Los percentiles ayudan a decidir si los umbrales propuestos para `hidden_gems_v3`, como `artist_popularity < 60` y `artist_followers < 500000`, son razonables.

In [16]:
numeric_summary = artists_typed.select(
    F.count("*").alias("n_rows"),

    F.min("followers").alias("followers_min"),
    F.expr("percentile_approx(followers, 0.50, 10000)").alias("followers_p50"),
    F.expr("percentile_approx(followers, 0.75, 10000)").alias("followers_p75"),
    F.expr("percentile_approx(followers, 0.90, 10000)").alias("followers_p90"),
    F.expr("percentile_approx(followers, 0.95, 10000)").alias("followers_p95"),
    F.expr("percentile_approx(followers, 0.99, 10000)").alias("followers_p99"),
    F.max("followers").alias("followers_max"),

    F.min("popularity").alias("popularity_min"),
    F.expr("percentile_approx(popularity, 0.50, 10000)").alias("popularity_p50"),
    F.expr("percentile_approx(popularity, 0.75, 10000)").alias("popularity_p75"),
    F.expr("percentile_approx(popularity, 0.90, 10000)").alias("popularity_p90"),
    F.expr("percentile_approx(popularity, 0.95, 10000)").alias("popularity_p95"),
    F.expr("percentile_approx(popularity, 0.99, 10000)").alias("popularity_p99"),
    F.max("popularity").alias("popularity_max")
)

numeric_summary.show(truncate=False, vertical= True)

[Stage 25:>                                                         (0 + 1) / 1]

-RECORD 0---------------------
 n_rows         | 1162095     
 followers_min  | 0.0         
 followers_p50  | 57.0        
 followers_p75  | 417.0       
 followers_p90  | 3014.0      
 followers_p95  | 10497.0     
 followers_p99  | 118813.0    
 followers_max  | 7.8900234E7 
 popularity_min | 0           
 popularity_p50 | 2           
 popularity_p75 | 13          
 popularity_p90 | 30          
 popularity_p95 | 39          
 popularity_p99 | 56          
 popularity_max | 100         



Interpretación rápida:

La mayoría de artistas tiene muy pocos seguidores.
followers tiene una distribución extremadamente desigual.
El 99% de artistas tiene menos de aproximadamente 118,813 seguidores.
Entonces 500,000 seguidores está por encima del p99. Es decir, ese umbral solo excluye artistas de la cola más masiva.
Para popularity, el p99 es 56, así que artist_popularity < 60 también excluye principalmente artistas muy visibles.

## Validación de valores inválidos

Después de revisar la distribución de las variables numéricas, se valida si existen valores fuera de los rangos esperados.

Los criterios usados son:

- `id` nulo o vacío;
- `name` nulo o vacío;
- `popularity` fuera del intervalo [0, 100];
- `followers` negativo;
- `followers` nulo.

Esta validación permite distinguir entre valores realmente inválidos y valores simplemente faltantes.

In [17]:
invalid_summary = artists_typed.select(
    F.sum((F.col("id").isNull() | (F.col("id") == "")).cast("int")).alias("id_null_or_empty"),

    F.sum((F.col("name").isNull() | (F.col("name") == "")).cast("int")).alias("name_null_or_empty"),

    F.sum(
        (
            F.col("popularity").isNotNull() &
            ((F.col("popularity") < 0) | (F.col("popularity") > 100))
        ).cast("int")
    ).alias("popularity_out_of_range"),

    F.sum(
        (
            F.col("followers").isNotNull() &
            (F.col("followers") < 0)
        ).cast("int")
    ).alias("followers_negative"),

    F.sum(F.col("followers").isNull().cast("int")).alias("followers_null")
)

invalid_summary.show(truncate=False)

[Stage 26:=====================>                                    (3 + 5) / 8]

+----------------+------------------+-----------------------+------------------+--------------+
|id_null_or_empty|name_null_or_empty|popularity_out_of_range|followers_negative|followers_null|
+----------------+------------------+-----------------------+------------------+--------------+
|0               |0                 |0                      |0                 |11            |
+----------------+------------------+-----------------------+------------------+--------------+



## Revisar genres_raw

In [18]:
artists_typed.select("genres_raw").distinct().show(30, truncate=False)

+------------------------------------------------------------+
|genres_raw                                                  |
+------------------------------------------------------------+
|['euro hi-nrg']                                             |
|['singaporean pop']                                         |
|['tropical house']                                          |
|['gauze pop']                                               |
|['malaysian mandopop']                                      |
|['ectofolk']                                                |
|['ai']                                                      |
|['goa trance', 'nitzhonot']                                 |
|['azeri rap']                                               |
|['trap chileno']                                            |
|['musica mapuche']                                          |
|['j-rap']                                                   |
|['psychedelic soul']                                  

### Conteo de artistas con y sin géneros

Después de revisar ejemplos de `genres_raw`, se observa que la columna contiene listas serializadas como texto. Algunos artistas tienen listas vacías `[]`, mientras que otros tienen uno o varios géneros.

Antes de convertir esta columna a un arreglo real de Spark, se calcula cuántos artistas tienen géneros registrados y cuántos no. Esto permite decidir si la variable puede ser útil para análisis posteriores.

In [19]:
genres_raw_summary = artists_typed.select(
    F.count("*").alias("n_artists"),
    F.sum((F.col("genres_raw") == "[]").cast("int")).alias("genres_empty_list"),
    F.sum((F.col("genres_raw") != "[]").cast("int")).alias("genres_non_empty"),
    F.avg((F.col("genres_raw") != "[]").cast("double")).alias("share_with_genres")
)

genres_raw_summary.show(truncate=False)

+---------+-----------------+----------------+------------------+
|n_artists|genres_empty_list|genres_non_empty|share_with_genres |
+---------+-----------------+----------------+------------------+
|1162095  |856500           |305595          |0.2629690343732655|
+---------+-----------------+----------------+------------------+



### Resultado del diagnóstico de géneros

El diagnóstico muestra que 305,595 artistas tienen al menos un género registrado, lo que representa aproximadamente el 26.3% del total. En contraste, 856,500 artistas tienen una lista vacía de géneros.

Esto indica que `genres` puede ser útil como variable descriptiva o auxiliar, pero no debe usarse como criterio principal de filtrado para `hidden_gems_v3`, ya que la mayoría de los artistas no cuenta con géneros registrados.

Por esta razón, se conservará la columna de géneros y se crearán variables auxiliares como `num_genres` y `has_genres`, pero la definición estricta de hidden gems se basará principalmente en popularidad y seguidores del artista.

In [20]:
from pyspark.sql.types import ArrayType, StringType

In [28]:
#PARSEO

In [21]:
from pyspark.sql.types import ArrayType, StringType

empty_string_array = F.array().cast(ArrayType(StringType()))

genres_inner = F.regexp_replace(
    F.regexp_replace(F.col("genres_raw"), r"^\[", ""),
    r"\]$",
    ""
)

genres_split = F.split(genres_inner, r",\s*")

genres_cleaned = F.transform(
    genres_split,
    lambda x: F.regexp_replace(
        F.trim(x),
        r"""^['"]|['"]$""",
        ""
    )
)

artists_with_genres = (
    artists_typed
    .withColumn(
        "genres",
        F.when(
            (F.col("genres_raw").isNull()) | (F.col("genres_raw") == "[]"),
            empty_string_array
        ).otherwise(genres_cleaned)
    )
    .withColumn(
        "genres",
        F.expr("filter(genres, x -> x is not null and trim(x) <> '')")
    )
    .withColumn("num_genres", F.size(F.col("genres")))
    .withColumn("has_genres", F.col("num_genres") > 0)
    .cache()
)

In [22]:
artists_with_genres.select(
    "id",
    "name",
    "genres_raw",
    "genres"
).show(20, truncate=False)

[Stage 35:>                                                         (0 + 1) / 1]

+----------------------+----------------------------------+-------------+-----------+
|id                    |name                              |genres_raw   |genres     |
+----------------------+----------------------------------+-------------+-----------+
|31Sb76IFcQBvnSTP0BrYOo|Bobby Diamond                     |[]           |[]         |
|30K9bUO2UVuecuttUdnVQp|The Quickest Way Out              |[]           |[]         |
|6eMnOoMBmtMesYaS8T2R5G|Wigan's Ovation                   |[]           |[]         |
|0Y3V9aEmAL78ePgFsS5FnD|Dianne Steinberg                  |[]           |[]         |
|2B0gsVlWByBbij8h6y2TFo|The Patterson Twins               |[]           |[]         |
|404qQMseMZFa7yxJFHAqeF|Tommy & Eddie                     |[]           |[]         |
|4j2stfqVSryQAeEuhUyZVa|Luv Co                            |[]           |[]         |
|1gOvuBeTA5WDWtzoDSLAhB|Lee Mcdonald aka: Cleveland Parker|[]           |[]         |
|76nrx0hcuOEP3R9ZgUeJbt|Jason Pike                    

## Creación de columnas auxiliares has_genres , num_genres

In [23]:
empty_string_array = F.array().cast(ArrayType(StringType()))

artists_with_genres = (
    artists_with_genres
    .withColumn(
        "genres",
        F.coalesce(F.col("genres"), empty_string_array)
    )
    .withColumn("num_genres", F.size(F.col("genres")))
    .withColumn("has_genres", F.col("num_genres") > 0)
    .cache()
)

In [24]:
artists_with_genres.select(
    "id",
    "name",
    "genres_raw",
    "genres",
    "num_genres",
    "has_genres"
).show(30, truncate=False)

[Stage 36:>                                                         (0 + 1) / 1]

+----------------------+----------------------------------+-------------+-----------+----------+----------+
|id                    |name                              |genres_raw   |genres     |num_genres|has_genres|
+----------------------+----------------------------------+-------------+-----------+----------+----------+
|31Sb76IFcQBvnSTP0BrYOo|Bobby Diamond                     |[]           |[]         |0         |false     |
|30K9bUO2UVuecuttUdnVQp|The Quickest Way Out              |[]           |[]         |0         |false     |
|6eMnOoMBmtMesYaS8T2R5G|Wigan's Ovation                   |[]           |[]         |0         |false     |
|0Y3V9aEmAL78ePgFsS5FnD|Dianne Steinberg                  |[]           |[]         |0         |false     |
|2B0gsVlWByBbij8h6y2TFo|The Patterson Twins               |[]           |[]         |0         |false     |
|404qQMseMZFa7yxJFHAqeF|Tommy & Eddie                     |[]           |[]         |0         |false     |
|4j2stfqVSryQAeEuhUyZVa|Luv 

In [25]:
#resumen
genres_parsed_summary = artists_with_genres.select(
    F.count("*").alias("n_artists"),
    F.sum((F.col("num_genres") == 0).cast("int")).alias("artists_without_genres"),
    F.sum((F.col("num_genres") > 0).cast("int")).alias("artists_with_genres"),
    F.avg((F.col("num_genres") > 0).cast("double")).alias("share_with_genres"),
    F.max("num_genres").alias("max_num_genres")
)

genres_parsed_summary.show(truncate=False)

[Stage 37:==================================================>       (7 + 1) / 8]

+---------+----------------------+-------------------+------------------+--------------+
|n_artists|artists_without_genres|artists_with_genres|share_with_genres |max_num_genres|
+---------+----------------------+-------------------+------------------+--------------+
|1162095  |856500                |305595             |0.2629690343732655|21            |
+---------+----------------------+-------------------+------------------+--------------+



## Construcción de `artists_clean`

Con base en los diagnósticos anteriores, se aplica una limpieza mínima y conservadora.

Las decisiones son:

- No se eliminan registros, porque todos los artistas tienen `id` válido y no hay duplicados.
- Se conservan los 11 casos con `followers` nulo como valores faltantes.
- Se conserva `popularity`, ya que todos sus valores están dentro del rango esperado [0, 100].
- Se conserva `genres_raw` como evidencia del dato original.
- Se conserva `genres` como arreglo estructurado de Spark.
- Se agregan `num_genres` y `has_genres` como variables auxiliares.

Esta limpieza es defendible porque no elimina información innecesariamente y deja preparada la tabla para análisis posteriores.

In [29]:
artists_clean = (
    artists_with_genres
    .select(
        "id",
        "name",
        "followers",
        "popularity",
        "genres_raw",
        "genres",
        "num_genres",
        "has_genres"
    )
    .cache()
)

In [30]:
artists_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- followers: double (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- genres: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- num_genres: integer (nullable = false)
 |-- has_genres: boolean (nullable = false)



In [31]:
artists_clean.show(20, truncate=False)

+----------------------+----------------------------------+---------+----------+-------------+-----------+----------+----------+
|id                    |name                              |followers|popularity|genres_raw   |genres     |num_genres|has_genres|
+----------------------+----------------------------------+---------+----------+-------------+-----------+----------+----------+
|31Sb76IFcQBvnSTP0BrYOo|Bobby Diamond                     |4.0      |0         |[]           |[]         |0         |false     |
|30K9bUO2UVuecuttUdnVQp|The Quickest Way Out              |3.0      |0         |[]           |[]         |0         |false     |
|6eMnOoMBmtMesYaS8T2R5G|Wigan's Ovation                   |75.0     |10        |[]           |[]         |0         |false     |
|0Y3V9aEmAL78ePgFsS5FnD|Dianne Steinberg                  |7.0      |0         |[]           |[]         |0         |false     |
|2B0gsVlWByBbij8h6y2TFo|The Patterson Twins               |33.0     |0         |[]           |[] 

In [32]:
# validemos que no se haya perdido algo
artists_clean_count = artists_clean.count()
artists_clean_unique_ids = artists_clean.select("id").distinct().count()

print(f"Filas en artists_clean: {artists_clean_count:,}")
print(f"IDs únicos en artists_clean: {artists_clean_unique_ids:,}")

[Stage 52:==================================================>       (7 + 1) / 8]

Filas en artists_clean: 1,162,095
IDs únicos en artists_clean: 1,162,095


In [33]:
ARTISTS_CLEAN_PATH = "../processed/artists_clean.parquet"

artists_clean.write.mode("overwrite").parquet(ARTISTS_CLEAN_PATH)

26/05/21 02:54:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [34]:
artists_clean_check = spark.read.parquet(ARTISTS_CLEAN_PATH)

print(f"Filas guardadas en artists_clean.parquet: {artists_clean_check.count():,}")
artists_clean_check.printSchema()

Filas guardadas en artists_clean.parquet: 1,162,095
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- followers: double (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- num_genres: integer (nullable = true)
 |-- has_genres: boolean (nullable = true)



## Construcción de `artists_ref`

A partir de `artists_clean`, se construye una tabla de referencia para unir información de artistas con la tabla de canciones.

La tabla `artists_ref` usa nombres explícitos para evitar ambigüedad durante los joins:

- `artist_id`
- `artist_name`
- `artist_popularity`
- `artist_followers`
- `artist_popularity_class`
- `artist_size_class`
- `genres`
- `num_genres`
- `has_genres`

Esta tabla será usada en `hidden_gems_v3` para evitar que canciones de artistas masivos sean clasificadas como hidden gems únicamente porque tienen baja popularidad a nivel track.

In [35]:
artists_ref = (
    artists_clean
    .withColumn(
        "artist_popularity_class",
        F.when(F.col("popularity").isNull(), F.lit("desconocida"))
         .when(F.col("popularity") < 30, F.lit("baja"))
         .when(F.col("popularity") < 60, F.lit("media"))
         .otherwise(F.lit("alta"))
    )
    .withColumn(
        "artist_size_class",
        F.when(F.col("followers").isNull(), F.lit("desconocido"))
         .when(F.col("followers") < 1_000, F.lit("micro"))
         .when(F.col("followers") < 10_000, F.lit("pequeño"))
         .when(F.col("followers") < 100_000, F.lit("mediano"))
         .when(F.col("followers") < 500_000, F.lit("grande"))
         .otherwise(F.lit("masivo"))
    )
    .select(
        F.col("id").alias("artist_id"),
        F.col("name").alias("artist_name"),
        F.col("popularity").alias("artist_popularity"),
        F.col("followers").alias("artist_followers"),
        "artist_popularity_class",
        "artist_size_class",
        "genres",
        "num_genres",
        "has_genres"
    )
    .cache()
)

In [36]:
artists_ref.printSchema()

root
 |-- artist_id: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_popularity: integer (nullable = true)
 |-- artist_followers: double (nullable = true)
 |-- artist_popularity_class: string (nullable = false)
 |-- artist_size_class: string (nullable = false)
 |-- genres: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- num_genres: integer (nullable = false)
 |-- has_genres: boolean (nullable = false)



In [38]:
artists_ref.show(20, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------
 artist_id               | 31Sb76IFcQBvnSTP0BrYOo             
 artist_name             | Bobby Diamond                      
 artist_popularity       | 0                                  
 artist_followers        | 4.0                                
 artist_popularity_class | baja                               
 artist_size_class       | micro                              
 genres                  | []                                 
 num_genres              | 0                                  
 has_genres              | false                              
-RECORD 1-----------------------------------------------------
 artist_id               | 30K9bUO2UVuecuttUdnVQp             
 artist_name             | The Quickest Way Out               
 artist_popularity       | 0                                  
 artist_followers        | 3.0                                
 artist_popularity_class | baja                        

In [40]:
#Artistas grandes
artists_ref.select(
    "artist_id",
    "artist_name",
    "artist_popularity",
    "artist_followers",
    "artist_popularity_class",
    "artist_size_class",
    "genres",
    "num_genres"
).orderBy(F.desc("artist_followers")).show(30, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------
 artist_id               | 6eUKZXaKkcviH0Ku9w2n3V                                               
 artist_name             | Ed Sheeran                                                           
 artist_popularity       | 92                                                                   
 artist_followers        | 7.8900234E7                                                          
 artist_popularity_class | alta                                                                 
 artist_size_class       | masivo                                                               
 genres                  | [pop, uk pop]                                                        
 num_genres              | 2                                                                    
-RECORD 1---------------------------------------------------------------------------------------
 artist_id               | 66C

In [41]:
popularity_class_counts = (
    artists_ref
    .groupBy("artist_popularity_class")
    .agg(
        F.count("*").alias("n_artists"),
        F.min("artist_popularity").alias("min_popularity"),
        F.max("artist_popularity").alias("max_popularity"),
        F.avg("artist_popularity").alias("avg_popularity")
    )
    .orderBy(
        F.when(F.col("artist_popularity_class") == "desconocida", 0)
         .when(F.col("artist_popularity_class") == "baja", 1)
         .when(F.col("artist_popularity_class") == "media", 2)
         .when(F.col("artist_popularity_class") == "alta", 3)
         .otherwise(4)
    )
)

popularity_class_counts.show(truncate=False)

+-----------------------+---------+--------------+--------------+------------------+
|artist_popularity_class|n_artists|min_popularity|max_popularity|avg_popularity    |
+-----------------------+---------+--------------+--------------+------------------+
|baja                   |1045689  |0             |29            |5.140516922335417 |
|media                  |108817   |30            |59            |39.880211731622815|
|alta                   |7589     |60            |100           |66.76966662274344 |
+-----------------------+---------+--------------+--------------+------------------+



### Resultado de la clasificación por popularidad

La clasificación por popularidad muestra una distribución altamente concentrada en la clase `baja`. La mayoría de los artistas tiene popularidad menor a 30, mientras que solo una fracción pequeña alcanza valores iguales o superiores a 60.

Esto es consistente con los percentiles calculados previamente, donde el percentil 99 de popularidad se ubicó alrededor de 56. Por lo tanto, usar `artist_popularity >= 60` como umbral de alta popularidad es una decisión estricta y defendible.

En el contexto de `hidden_gems_v3`, esta variable permite excluir canciones de artistas globalmente visibles aunque la popularidad específica del track sea baja.

In [42]:
size_class_counts = (
    artists_ref
    .groupBy("artist_size_class")
    .agg(
        F.count("*").alias("n_artists"),
        F.min("artist_followers").alias("min_followers"),
        F.max("artist_followers").alias("max_followers"),
        F.avg("artist_followers").alias("avg_followers")
    )
    .orderBy(
        F.when(F.col("artist_size_class") == "desconocido", 0)
         .when(F.col("artist_size_class") == "micro", 1)
         .when(F.col("artist_size_class") == "pequeño", 2)
         .when(F.col("artist_size_class") == "mediano", 3)
         .when(F.col("artist_size_class") == "grande", 4)
         .when(F.col("artist_size_class") == "masivo", 5)
         .otherwise(6)
    )
)

size_class_counts.show(truncate=False)

+-----------------+---------+-------------+-------------+------------------+
|artist_size_class|n_artists|min_followers|max_followers|avg_followers     |
+-----------------+---------+-------------+-------------+------------------+
|desconocido      |11       |NULL         |NULL         |NULL              |
|micro            |962390   |0.0          |999.0        |122.32595413501802|
|pequeño          |139840   |1000.0       |9999.0       |3273.332079519451 |
|mediano          |46585    |10000.0      |99995.0      |30579.857486315337|
|grande           |9539     |100000.0     |499804.0     |216086.74599014572|
|masivo           |3730     |500222.0     |7.8900234E7  |2095451.5801608579|
+-----------------+---------+-------------+-------------+------------------+



### Resultado de la clasificación por seguidores

La clasificación por seguidores muestra una distribución de cola larga. La mayoría de los artistas pertenece a la clase `micro`, con menos de 1,000 seguidores. En contraste, solo 3,730 artistas aparecen en la categoría `masivo`, definida como artistas con 500,000 seguidores o más.

Esto confirma que el umbral de 500,000 seguidores es conservador: no elimina artistas medianos o grandes de forma agresiva, sino que identifica únicamente a artistas claramente masivos.

En `hidden_gems_v3`, este criterio ayuda a evitar falsos positivos, es decir, canciones con baja popularidad a nivel track pero pertenecientes a artistas globalmente conocidos.

In [43]:
#validar que artists_ref conserva todas las filas
artists_ref_count = artists_ref.count()
artists_ref_unique_ids = artists_ref.select("artist_id").distinct().count()

print(f"Filas en artists_ref: {artists_ref_count:,}")
print(f"IDs únicos en artists_ref: {artists_ref_unique_ids:,}")

[Stage 76:=======>                                                  (1 + 7) / 8]

Filas en artists_ref: 1,162,095
IDs únicos en artists_ref: 1,162,095


In [44]:
artists_ref.printSchema()

root
 |-- artist_id: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_popularity: integer (nullable = true)
 |-- artist_followers: double (nullable = true)
 |-- artist_popularity_class: string (nullable = false)
 |-- artist_size_class: string (nullable = false)
 |-- genres: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- num_genres: integer (nullable = false)
 |-- has_genres: boolean (nullable = false)



## Guardamos artistas_ref

In [45]:
ARTISTS_REF_PATH = "../processed/artists_ref.parquet"

artists_ref.write.mode("overwrite").parquet(ARTISTS_REF_PATH)

26/05/21 03:17:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [46]:
#verificamos
artists_ref_check = spark.read.parquet(ARTISTS_REF_PATH)

print(f"Filas guardadas en artists_ref.parquet: {artists_ref_check.count():,}")
artists_ref_check.printSchema()

Filas guardadas en artists_ref.parquet: 1,162,095
root
 |-- artist_id: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_popularity: integer (nullable = true)
 |-- artist_followers: double (nullable = true)
 |-- artist_popularity_class: string (nullable = true)
 |-- artist_size_class: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- num_genres: integer (nullable = true)
 |-- has_genres: boolean (nullable = true)



In [47]:
print(
    "IDs únicos en artists_ref_check:",
    artists_ref_check.select("artist_id").distinct().count()
)

[Stage 87:==============>                                           (2 + 6) / 8]

IDs únicos en artists_ref_check: 1162095


### Guardamos diagnosticos

In [48]:
ARTISTS_DIAG_DIR = "../processed/diagnostics/artists"

popularity_class_counts.write.mode("overwrite").parquet(
    f"{ARTISTS_DIAG_DIR}/popularity_class_counts.parquet"
)

size_class_counts.write.mode("overwrite").parquet(
    f"{ARTISTS_DIAG_DIR}/size_class_counts.parquet"
)

numeric_summary.write.mode("overwrite").parquet(
    f"{ARTISTS_DIAG_DIR}/numeric_summary.parquet"
)

nulls_summary.write.mode("overwrite").parquet(
    f"{ARTISTS_DIAG_DIR}/nulls_summary.parquet"
)

invalid_summary.write.mode("overwrite").parquet(
    f"{ARTISTS_DIAG_DIR}/invalid_summary.parquet"
)

genres_parsed_summary.write.mode("overwrite").parquet(
    f"{ARTISTS_DIAG_DIR}/genres_parsed_summary.parquet"
)

## Conclusión

La tabla `artists_ref` queda lista para enriquecer `tracks_scored_v2`.

Los diagnósticos muestran que `artists.csv` es confiable para joins: no hay IDs nulos, no hay IDs duplicados y la popularidad está dentro del rango esperado [0, 100]. El único faltante relevante son 11 registros con `followers` nulo, los cuales se conservan como valores faltantes.

La clasificación por popularidad y seguidores permite distinguir artistas de baja, media y alta visibilidad. Esto será usado en `hidden_gems_v3` para evitar falsos positivos: canciones con baja popularidad a nivel track, pero pertenecientes a artistas globalmente masivos.

La siguiente etapa será unir `tracks_scored_v2` con `artists_ref` y calcular, por canción:

- `max_artist_popularity`
- `max_artist_followers`
- `artist_names_ref`
- `num_artists_joined`